# Trino and Spark federation
Run from the training Jupyter workbench. The lab creates uniquely named tables, checks every directed transfer, and leaves evidence for inspection. Use a 10 GB Minikube limit for the expanded stack.

In [ ]:
%run /workspace/pyspark/federation/federation_lab.py

In [ ]:
import json
from pathlib import Path
result = json.loads(Path("/workspace/pyspark/federation/results.json").read_text())
print(result["status"], len(result["checks"]), result["run_id"])
result["checks"]

## Query the actual tables from this run
The same SQL can be pasted into the Trino CLI after substituting the displayed table name. Spark access and JDBC configurations are in `federation_lab.py`.

In [ ]:
import trino
table = "r" + result["run_id"] + "_spark_seed"
connection = trino.dbapi.connect(host="trino", port=8080, user="team")
cursor = connection.cursor()
query = f"""SELECT m.id, m.name, p.city, h.city AS hive_city, i.city AS hdfs_iceberg_city, s.city AS s3_iceberg_city
FROM mysql.training.{table} m
JOIN postgresql.public.{table} p ON m.id = p.id
JOIN hive.federation_hive.{table} h ON m.id = h.id
JOIN iceberg_hive.federation_ice_hdfs.{table} i ON m.id = i.id
JOIN polaris.federation_polaris.{table} s ON m.id = s.id
ORDER BY m.id"""
print(query)
cursor.execute(query)
print(cursor.fetchall())
cursor.close()
connection.close()